# Enterprise LLM Confidence Calibration Benchmark Suite
### Evaluating 10 Confidence Scoring Frameworks for Resume Parsing using Google `google-genai` SDK

---

## Summary
In production AI resume parsing pipelines, raw LLM completions often exhibit severe **overconfidence** or **probability saturation**. Standard uncalibrated confidence leads to automated misclassifications or unnecessary human auditor workload.

This notebook demonstrates how to build a custom **Pydantic `BaseModel` for PDF Resume parsing**, evaluate, calibrate, and compare **9 distinct confidence score frameworks**, and route candidate extractions reliably between automated processing (`AUTOMATE`) and human audit (`FLAG_FOR_HUMAN_REVIEW`).


## Load Libraries

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import random
import os
from dotenv import load_dotenv

from google import genai

from pydantic import BaseModel, Field

# Ensure src is in python path
sys.path.append("..")
sys.path.append(".")

from src.ingestion import ResumeIngestor
from src.schema import DomainRole, SeniorityLevel, AuditDecision

## Gemini

In [2]:
# Load env vars from .env
load_dotenv(override=True)

use_vertex = os.getenv("USE_VERTEX_AI", "false").lower() in ("true", "1")

if use_vertex:
    project = os.getenv("GOOGLE_CLOUD_PROJECT")
    location = os.getenv("GOOGLE_CLOUD_LOCATION")
    model = os.getenv("VERTEX_GEMINI_MODEL")
    print(f"Initializing Gemini Client in Vertex AI Mode (Project: {project}, Region: {location}, Model: {model})")
    client = genai.Client(vertexai=True, project=project, location=location)
else:
    api_key = os.getenv("GEMINI_API_KEY")
    model = os.getenv("API_KEY_GEMINI_MODEL")
    print(f"Initializing Gemini Client in AI Studio API Key Mode (Model: {model})")
    client = genai.Client(api_key=api_key)


Initializing Gemini Client in Vertex AI Mode (Project: gemini-sandbox-446215, Region: us-central1, Model: gemini-2.5-flash)


## 1. Custom Resume BaseModel Definition & Data Ingestion
We define a custom Pydantic `BaseModel` (`ResumeSchema`) to extract candidate metadata from PDF resumes. All 10 methods accept this custom schema (`schema=ResumeSchema`) and custom prompt instructions.


In [3]:
# Define Custom Resume BaseModel Schema for Candidate Parsing
class ResumeSchema(BaseModel):
    candidate_name: str = Field(default="Unknown Candidate", description="Full name of the candidate")
    domain_role: DomainRole = Field(default=DomainRole.OTHER, description="Primary candidate professional role classification")
    seniority_level: SeniorityLevel = Field(default=SeniorityLevel.OTHER, description="Candidate seniority tier")
    years_of_experience: float = Field(default=0.0, description="Estimated total years of professional experience")
    primary_skills: list[str] = Field(default_factory=list, description="Top technical and professional skills extracted")
    education_level: str = Field(default="Unspecified", description="Highest educational degree or background")
    key_claims: list[str] = Field(default_factory=list, description="Atomic factual claims extracted from resume")


In [4]:
# Ingest sample PDF resume from dataset
data_dir = Path("data") if Path("data").exists() else Path("../data")
ingestor = ResumeIngestor(data_directory=data_dir)
pdf_files = ingestor.scan_resumes()
print(f"Found {len(pdf_files)} PDF resume files in dataset directory.\n")

Found 20 PDF resume files in dataset directory.



In [5]:
sample_pdf = random.choice(pdf_files)
sample_text, sample_bytes = ingestor.process_pdf(sample_pdf)
print(f"Sample Resume File: {sample_pdf.name}")
print("=" * 60)
print(sample_text[:500] + "\n...")

Sample Resume File: 11551946.pdf
--- Page 1 ---
SENIOR BUSINESS DEVELOPMENT MANAGER
Summary
Business development professional experienced in sales management, marketing and applications engineering of filtration and drying systems to the
bio-pharmaceuticals, fine chemicals, agro-chemicals, and other industries.
Highlights
Accomplishments
Supercharged sales of high-valued filtration systems to customers in the bio-pharmaceuticals, specialty chemicals, agro-chemicals, and other
industries using consultative selling techniques, to
...


## 2. Individual Confidence Score Method Frameworks
Below, each of the 9 confidence calibration engines is presented individually with a mathematical breakdown of its algorithm, followed by execution demonstrating evaluation with our custom `ResumeSchema`.


### Method 1: Native Token LogProb Method (`NativeLogProbMethod`)

#### Algorithm & Mathematical Foundation
Configures `response_logprobs=True` and `logprobs=5` via `GenerateContentConfig`. Calculates joint sequence confidence as the geometric mean token probability across generated output tokens:

$$\text{Confidence} = \exp\left( \frac{1}{N} \sum_{i=1}^N \text{logprob}_i \right)$$

* **Pros:** Native token probability directly from LLM decoding. Fast (< 5ms overhead).
* **Cons:** Vulnerable to RLHF probability saturation where LLMs assign artificially high probabilities.


In [ ]:
from src.calibration import NativeLogProbMethod

# Instantiate with client
engine_1 = NativeLogProbMethod(client=client)

# Evaluate on Custom Resume Schema
res_1 = engine_1.evaluate(sample_text, schema=ResumeSchema, pdf_bytes=sample_bytes)

print(f"Engine: {res_1.method_name}")
print(f"Target Schema: {res_1.metadata['target_schema']}")
print(f"Candidate Name: {getattr(res_1.extraction, 'candidate_name', 'N/A')}")
print(f"Extracted Domain Role: {getattr(res_1.extraction, 'domain_role', DomainRole.OTHER).value}")
print(f"Seniority Level: {getattr(res_1.extraction, 'seniority_level', SeniorityLevel.OTHER).value}")
print(f"Raw Confidence: {res_1.raw_confidence:.4f}")
print(f"Calibrated Confidence: {res_1.calibrated_confidence:.4f}")
print(f"Audit Decision: {res_1.audit_decision.value}")
print(f"Mean LogProb: {res_1.metadata['mean_logprob']:.4f}")
print(f"Latency: {res_1.latency_ms:.2f} ms")

Engine: Native Token LogProb
Target Schema: ResumeSchema
Candidate Name: null
Extracted Domain Role: Business Development
Seniority Level: Lead / Management
Raw Confidence: 0.9838
Calibrated Confidence: 0.9838
Audit Decision: AUTOMATE
Mean LogProb: -0.0164
Latency: 15580.66 ms


### Method 2: LogProb Delta Method (`LogProbDeltaMethod`)

#### Algorithm & Mathematical Foundation
Isolates top-token dominance margin at critical decision tokens to eliminate probability saturation. Computes the logprob difference between top 1 and top 2 alternative candidates:

$$\Delta = \text{logprob}_{\text{top\_1}} - \text{logprob}_{\text{top\_2}}$$
$$\text{Confidence} = \sigma(\Delta) = \frac{1}{1 + e^{-\Delta}}$$

* **Pros:** Measures decision margin; robust against uniform model confidence shifts.
* **Cons:** Requires accessing token alternative logprob distribution via API.


In [7]:
from src.calibration import LogProbDeltaMethod

engine_2 = LogProbDeltaMethod(client=client)
res_2 = engine_2.evaluate(sample_text, schema=ResumeSchema, pdf_bytes=sample_bytes)

print(f"Engine: {res_2.method_name}")
print(f"Target Schema: {res_2.metadata['target_schema']}")
print(f"Raw Confidence: {res_2.raw_confidence:.4f}")
print(f"Calibrated Confidence: {res_2.calibrated_confidence:.4f}")
print(f"Audit Decision: {res_2.audit_decision.value}")
print(f"Top 1 vs Top 2 Delta: {res_2.metadata['mean_delta']:.4f}")
print(f"Latency: {res_2.latency_ms:.2f} ms")


Engine: LogProb Delta
Target Schema: ResumeSchema
Raw Confidence: 1.0000
Calibrated Confidence: 1.0000
Audit Decision: AUTOMATE
Top 1 vs Top 2 Delta: 14.0124
Latency: 14562.98 ms


### Method 3: Post-Hoc Temperature Scaling Method (`TemperatureScalingMethod`)

#### Algorithm & Mathematical Foundation
Fits a single temperature parameter $T > 1$ on raw logit/logprob values using Negative Log-Likelihood (NLL) optimization on a validation split via `scipy.optimize.minimize`:

$$P_{\text{calibrated}}(i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

* **Pros:** Preserves classification ranking while adjusting probability spread. Highly effective for expected calibration error reduction.
* **Cons:** Requires a labeled validation set to fit temperature parameter $T$.


In [8]:
from src.calibration import TemperatureScalingMethod

engine_3 = TemperatureScalingMethod(client=client, temperature=1.35)

# Fit temperature on sample validation split
val_logits = np.array([2.5, 1.2, -0.5, 3.1, 0.8])
val_labels = np.array([1, 1, 0, 1, 0])
fitted_T = engine_3.fit(val_logits, val_labels)
print(f"Fitted Optimal Temperature T: {fitted_T:.4f}")

res_3 = engine_3.evaluate(sample_text, schema=ResumeSchema, pdf_bytes=sample_bytes)

print(f"Engine: {res_3.method_name}")
print(f"Target Schema: {res_3.metadata['target_schema']}")
print(f"Raw Confidence: {res_3.raw_confidence:.4f}")
print(f"Calibrated Confidence: {res_3.calibrated_confidence:.4f}")
print(f"Audit Decision: {res_3.audit_decision.value}")
print(f"Latency: {res_3.latency_ms:.2f} ms")


Fitted Optimal Temperature T: 0.8178
Engine: Temperature Scaling
Target Schema: ResumeSchema
Raw Confidence: 0.9838
Calibrated Confidence: 0.9802
Audit Decision: AUTOMATE
Latency: 15766.40 ms


### Method 4: Platt Scaling Logistic Method (`PlattScalingMethod`)

#### Algorithm & Mathematical Foundation
Trains a 1D `LogisticRegression` model mapping logprob delta $\Delta$ (or raw confidence) against empirical correctness on a validation fold:

$$P(y=1 \mid \Delta) = \frac{1}{1 + \exp(A \cdot \Delta + B)}$$

* **Pros:** Supervised calibration mapping model certainty to true empirical accuracy.
* **Cons:** Parametric assumption of sigmoid relationship.


In [9]:
from src.calibration import PlattScalingMethod

engine_4 = PlattScalingMethod(client=client)

# Fit Platt scaling parameters on validation data
val_deltas = np.array([2.4, 0.5, 1.8, 3.1, 0.2])
val_correctness = np.array([1, 0, 1, 1, 0])
engine_4.fit(val_deltas, val_correctness)

res_4 = engine_4.evaluate(sample_text, schema=ResumeSchema, pdf_bytes=sample_bytes)

print(f"Engine: {res_4.method_name}")
print(f"Target Schema: {res_4.metadata['target_schema']}")
print(f"Raw Confidence: {res_4.raw_confidence:.4f}")
print(f"Calibrated Confidence: {res_4.calibrated_confidence:.4f}")
print(f"Fitted Slope A: {res_4.metadata['platt_slope']:.4f}")
print(f"Fitted Intercept B: {res_4.metadata['platt_intercept']:.4f}")
print(f"Audit Decision: {res_4.audit_decision.value}")
print(f"Latency: {res_4.latency_ms:.2f} ms")


Engine: Platt Scaling Logistic
Target Schema: ResumeSchema
Raw Confidence: 1.0000
Calibrated Confidence: 1.0000
Fitted Slope A: 1.1109
Fitted Intercept B: -1.2274
Audit Decision: AUTOMATE
Latency: 13649.68 ms


### Method 5: Self-Consistency & Sampling Agreement Method (`SelfConsistencyMethod`)

#### Algorithm & Mathematical Foundation
Generates $N=5$ parallel completions at non-zero temperature $T=0.7$. Clusters output extractions semantically and computes the agreement ratio:

$$\text{Confidence} = \frac{\text{Majority Cluster Size}}{N}$$

* **Pros:** Highly robust; eliminates single-sample stochastic variance without logprob dependencies.
* **Cons:** Increases API call volume and latency by $N\times$.


In [10]:
from src.calibration import SelfConsistencyMethod

engine_5 = SelfConsistencyMethod(client=client, num_samples=5, sample_temperature=0.7)
res_5 = engine_5.evaluate(sample_text, schema=ResumeSchema, pdf_bytes=sample_bytes)

print(f"Engine: {res_5.method_name}")
print(f"Target Schema: {res_5.metadata['target_schema']}")
print(f"Calibrated Confidence (Agreement Ratio): {res_5.calibrated_confidence:.4f}")
print(f"Majority Count: {res_5.metadata['majority_count']} / {res_5.metadata['num_samples_successful']}")
print(f"Audit Decision: {res_5.audit_decision.value}")
print(f"Latency: {res_5.latency_ms:.2f} ms")


Engine: Self-Consistency & Agreement
Target Schema: ResumeSchema
Calibrated Confidence (Agreement Ratio): 1.0000
Majority Count: 5 / 5
Audit Decision: AUTOMATE
Latency: 95682.11 ms


### Method 6: Structured Self-Assessment Method (`VerbalizedConfidenceMethod`)

#### Algorithm & Mathematical Foundation
Enforces a Pydantic JSON schema via `response_schema` and `response_mime_type="application/json"`. Requests explicit step-by-step `reasoning`, `extracted_class`, and `verbalized_confidence_score` $S_{\text{verb}} \in [0.0, 1.0]$.

* **Pros:** Single API call providing step-by-step rationale alongside confidence score.
* **Cons:** LLMs tend to verbalize overconfident numbers without logprob groundings.


In [11]:
from src.calibration import VerbalizedConfidenceMethod

engine_6 = VerbalizedConfidenceMethod(client=client)
res_6 = engine_6.evaluate(sample_text, schema=ResumeSchema, pdf_bytes=sample_bytes)

print(f"Engine: {res_6.method_name}")
print(f"Target Schema: {res_6.metadata['target_schema']}")
print(f"Verbalized Confidence Score: {res_6.calibrated_confidence:.4f}")
print(f"LLM Reasoning: {res_6.metadata['reasoning']}")
print(f"Audit Decision: {res_6.audit_decision.value}")
print(f"Latency: {res_6.latency_ms:.2f} ms")


Engine: Structured Verbalized Confidence
Target Schema: ResumeSchema
Verbalized Confidence Score: 0.9500
LLM Reasoning: The document is a resume, which is typically well-structured. I will extract information from the 'Experience', 'Education', and 'Skills' sections. For 'Experience', I will identify each job entry by its title and dates, then extract the company name, location, and bulleted responsibilities. For 'Education', I will extract the university, degree, major, and graduation date. For 'Skills', I will parse the comma-separated list provided. The structure of the resume makes it easy to identify these distinct sections and their corresponding data points.
Audit Decision: AUTOMATE
Latency: 17225.44 ms


### Method 7: Continuous Numerical Prompting Method (`ContinuousPromptingMethod`)

#### Algorithm & Mathematical Foundation
Bypasses strict single-token enum constraints by prompting the LLM for detailed chain-of-thought rationale followed by a continuous integer rating $R \in [0, 100]$. Normalizes score to $[0.0, 1.0]$:

$$\text{Confidence} = \frac{R}{100.0}$$

* **Pros:** Continuous granular scale reduces quantization error of discrete logprobs.
* **Cons:** Model-dependent calibration curve; susceptible to prompt wording.


In [12]:
from src.calibration import ContinuousPromptingMethod

engine_7 = ContinuousPromptingMethod(client=client)
res_7 = engine_7.evaluate(sample_text, schema=ResumeSchema, pdf_bytes=sample_bytes)

print(f"Engine: {res_7.method_name}")
print(f"Target Schema: {res_7.metadata['target_schema']}")
print(f"Raw Rating (0-100): {res_7.metadata['rating_0_to_100']:.1f}")
print(f"Normalized Confidence: {res_7.calibrated_confidence:.4f}")
print(f"Chain of Thought: {res_7.metadata['chain_of_thought_rationale']}")
print(f"Audit Decision: {res_7.audit_decision.value}")
print(f"Latency: {res_7.latency_ms:.2f} ms")


Engine: Continuous Numerical Prompting
Target Schema: ResumeSchema
Raw Rating (0-100): 98.0
Normalized Confidence: 0.9800
Chain of Thought: The document is a well-structured resume, making it straightforward to identify and extract key sections such as experience, education, skills, and additional information. Job titles, company names, dates, and responsibilities are clearly delineated. Education details including university, degree, major, and graduation date are explicit. The skills section is presented as a clear, comma-separated list. The 'Company Name City, State' is a placeholder in the original text, which has been extracted as is. The only potential ambiguity is the repetition of 'English English' and 'Media and Society Media and Society' in the education section, which I've handled by assuming it's a single entry.
Audit Decision: AUTOMATE
Latency: 22128.22 ms


### Method 8: Grounding & Context Alignment Method (`GroundingAlignmentMethod`)

#### Algorithm & Mathematical Foundation
Extracts atomic factual claims from the extraction (e.g. "Senior Data Scientist with 6 years experience") and performs Natural Language Inference (NLI) verification against source text:

$$\text{Grounding Score} = \frac{\text{Supported Claims}}{\text{Total Claims}}$$

* **Pros:** Direct measure of factual hallucination risk and source context alignment.
* **Cons:** Incurred overhead for claim extraction and NLI verification pass.


In [13]:
from src.calibration import GroundingAlignmentMethod

engine_8 = GroundingAlignmentMethod(client=client)
res_8 = engine_8.evaluate(sample_text, schema=ResumeSchema, pdf_bytes=sample_bytes)

print(f"Engine: {res_8.method_name}")
print(f"Target Schema: {res_8.metadata['target_schema']}")
print(f"Supported Claims: {res_8.metadata['supported_claims']} / {res_8.metadata['total_claims']}")
print(f"Grounding Confidence Score: {res_8.calibrated_confidence:.4f}")
print(f"Audit Decision: {res_8.audit_decision.value}")
print(f"Latency: {res_8.latency_ms:.2f} ms")


Engine: Grounding & Alignment
Target Schema: ResumeSchema
Supported Claims: 26 / 26
Grounding Confidence Score: 1.0000
Audit Decision: AUTOMATE
Latency: 45965.69 ms


### Method 9: Two-Tier Evaluator Method (`LLMAsAJudgeMethod`)

#### Algorithm & Mathematical Foundation
Submits primary candidate extractions to a secondary evaluator model (`gemini-2.5-flash`) with a rubric for precision, hallucination risk, and completeness (1.0–5.0 scale), normalizing to $[0.0, 1.0]$:

$$\text{Normalized Score} = \frac{\text{Precision} + \text{Hallucination} + \text{Completeness}}{15.0}$$

* **Pros:** Comprehensive qualitative evaluation incorporating multi-dimension rubrics.
* **Cons:** Requires secondary LLM inference call; potential judge bias.


In [14]:
from src.calibration import LLMAsAJudgeMethod

engine_9 = LLMAsAJudgeMethod(client=client)
res_9 = engine_9.evaluate(sample_text, schema=ResumeSchema, pdf_bytes=sample_bytes)

print(f"Engine: {res_9.method_name}")
print(f"Target Schema: {res_9.metadata['target_schema']}")
print(f"Judge Model: {res_9.metadata['judge_model']}")
print(f"Precision Score (1-5): {res_9.metadata['precision_score']:.1f}")
print(f"Hallucination Score (1-5): {res_9.metadata['hallucination_score']:.1f}")
print(f"Completeness Score (1-5): {res_9.metadata['completeness_score']:.1f}")
print(f"Calibrated Confidence: {res_9.calibrated_confidence:.4f}")
print(f"Judge Rationale: {res_9.metadata['justification']}")
print(f"Audit Decision: {res_9.audit_decision.value}")
print(f"Latency: {res_9.latency_ms:.2f} ms")


Engine: LLM-as-a-Judge
Target Schema: ResumeSchema
Judge Model: gemini-2.5-flash
Precision Score (1-5): 5.0
Hallucination Score (1-5): 5.0
Completeness Score (1-5): 5.0
Calibrated Confidence: 1.0000
Judge Rationale: The extraction demonstrates excellent precision, with all fields accurately reflecting the source text. There are no hallucinations, as every piece of information can be directly verified against the source. The completeness is also very high, particularly in the 'key_claims' which capture nearly all significant achievements and responsibilities from the experience section. Derived fields like 'domain_role', 'seniority_level', and 'primary_skills' are well-inferred and representative of the source content.
Audit Decision: AUTOMATE
Latency: 31921.08 ms


### Method 10: Structured Self-Assessment + Platt Scaling Method (`StructuredSelfAssessmentPlattMethod`)

**Logprob-Independent Calibration Strategy for Gemini > 3.0 Models**

For newer Gemini models (e.g., Gemini 3.0 Flash, Gemini 3.5, Gemini 3.6 Flash/Pro), token logprobabilities (`response_logprobs`) are unavailable or disabled. The **Structured Self-Assessment + Platt Scaling Method** provides a robust, logprob-independent calibration solution:

1. **Structured Self-Assessment**: Prompts Gemini for structured extraction along with chain-of-thought rationale, a categorical confidence rating (`VERY_LOW`, `LOW`, `MEDIUM`, `HIGH`, `VERY_HIGH`), and a continuous numerical confidence rating in $[0.0, 1.0]$.
2. **Logit Transformation**: Converts raw self-assessment probability $p_{\text{raw}}$ to logit space:
   $$z = \text{logit}(p_{\text{raw}}) = \ln\left(\frac{p_{\text{raw}}}{1 - p_{\text{raw}}}\right)$$
3. **Platt Scaling**: Applies logistic calibration scaling fitted on empirical ground-truth validation data via NLL loss minimization:
   $$P_{\text{calibrated}} = \sigma(a \cdot z + b) = \frac{1}{1 + e^{-(a \cdot z + b)}}$$

This method includes a `.fit(val_scores, val_labels)` helper to optimize slope parameter $a$ and intercept parameter $b$.


In [6]:
from src.calibration import StructuredSelfAssessmentPlattMethod

# 1. Instantiate engine for Gemini > 3.0 logprob-free calibration
engine_10 = StructuredSelfAssessmentPlattMethod(client=client, a=1.0, b=0.0)

# 2. Fit Platt scaling parameters (slope a, intercept b) on validation split
val_self_assess_scores = np.array([0.92, 0.85, 0.70, 0.35, 0.15])
val_ground_truth_labels = np.array([1, 1, 1, 0, 0])
fitted_a, fitted_b = engine_10.fit(val_self_assess_scores, val_ground_truth_labels)

# 3. Evaluate structured self-assessment + Platt scaling on sample PDF resume
res_10 = engine_10.evaluate(sample_text, schema=ResumeSchema, pdf_bytes=sample_bytes)

print(f"Engine: {res_10.method_name}")
print(f"Target Schema: {res_10.metadata['target_schema']}")
print(f"Candidate Name: {getattr(res_10.extraction, 'candidate_name', 'N/A')}")
print(f"Extracted Domain Role: {getattr(res_10.extraction, 'domain_role', DomainRole.OTHER).value}")
print(f"Seniority Level: {getattr(res_10.extraction, 'seniority_level', SeniorityLevel.OTHER).value}")
print(f"Self-Assessed Level: {res_10.metadata.get('confidence_level', 'N/A')}")
print(f"Raw Self-Assessment Score: {res_10.raw_confidence:.4f}")
print(f"Raw Logit (z): {res_10.metadata.get('raw_logit', 0.0):.4f}")
print(f"Calibrated Logit (a*z + b): {res_10.metadata.get('calibrated_logit', 0.0):.4f}")
print(f"Calibrated Confidence: {res_10.calibrated_confidence:.4f}")
print(f"Fitted Platt Scale (a): {res_10.metadata['platt_a']:.4f}")
print(f"Fitted Platt Bias (b): {res_10.metadata['platt_b']:.4f}")
print(f"Audit Routing Decision: {res_10.audit_decision.value}")
print(f"Execution Latency: {res_10.latency_ms:.2f} ms")


Engine: Structured Self-Assessment + Platt Scaling
Target Schema: ResumeSchema
Candidate Name: Unknown Candidate
Extracted Domain Role: Other
Seniority Level: Other
Self-Assessed Level: Very High
Raw Self-Assessment Score: 0.9000
Raw Logit (z): 2.1972
Calibrated Logit (a*z + b): 30.5637
Calibrated Confidence: 1.0000
Fitted Platt Scale (a): 14.6434
Fitted Platt Bias (b): -1.6111
Audit Routing Decision: AUTOMATE
Execution Latency: 24789.36 ms
